## 1. Pipeline Banner

In [1]:
print("""
=========================================================
      ETL PIPELINE – WooCommerce (v1.0)
=========================================================
SOURCE → BRONZE → SILVER → GOLD
Star Schema:   dim_order, dim_time, dim_staff
                   +  fact_order_items
                   +  fact_orders_time
                   +  fact_staff_orders
=========================================================
""")


      PIPELINE ETL – WooCommerce (v1.0)
SOURCE → BRONZE → SILVER → GOLD
Modelo Estrella: dim_order, dim_time, dim_staffpipeline_woo
                   +  fact_order_items
                   +  fact_orders_time
                   +  fact_staff_orders



## 2. Global Imports

In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

from config.paths import PATHS, PATHS_PATH
from utils.io import read_csv, read_parquet, write_parquet, list_files

## 3. Ingestion → BRONZE

In [3]:
from pipelines.ingestion import ingest_all_source_csv

print("=== INGESTION (SOURCE → BRONZE) ===")
ingested = ingest_all_source_csv(verbose=True)
print("Ingested:", ingested)

=== INGESTA (SOURCE → BRONZE) ===
[BRONZE] Source activo: source\2026\03\06
[BRONZE] Encontrados 4 fichero(s) CSV
[BRONZE] Ingestando: BRONZE_posts_with_hours_20260306.csv → bronze\2026\03\06\post_with_hours.parquet
[BRONZE] Ingestando: BRONZE_woocommerce_order_itemmeta_202503180953_20260306.csv → bronze\2026\03\06\order_itemmeta.parquet
[BRONZE] Ingestando: BRONZE_woocommerce_order_items_202503180953_20260306.csv → bronze\2026\03\06\order_items.parquet
[BRONZE] Ingestando: ORDERS_WITH_STAFF_ASSIGNMENT_20260306.csv → bronze\2026\03\06\orders_with_staff_assignment.parquet
[BRONZE] Ingesta completa. Total: 4 → ['post_with_hours', 'order_itemmeta', 'order_items', 'orders_with_staff_assignment']
Ingestado: ['post_with_hours', 'order_itemmeta', 'order_items', 'orders_with_staff_assignment']


In [4]:
# sanity check — verify paths and files are in place
from importlib import reload
import config.paths as paths
paths = reload(paths)

from config.paths import PATHS
from utils.io import list_files
from pathlib import Path

print("PATHS:", PATHS)  # check effective paths

src_dir = Path(PATHS["source"])
print("Source folder exists?:", src_dir.exists())
print("CSVs found in source:", [p.name for p in list_files(src_dir, "*.csv")] + [p.name for p in list_files(src_dir, "*.CSV")])

PATHS: {'source': 'source\\2026\\03\\06', 'bronze': 'bronze\\2026\\03\\06', 'silver': 'silver\\2026\\03\\06', 'gold': 'gold\\2026\\03\\06'}
Existe source?: True
CSV en source: ['BRONZE_posts_with_hours_20260306.csv', 'BRONZE_woocommerce_order_itemmeta_202503180953_20260306.csv', 'BRONZE_woocommerce_order_items_202503180953_20260306.csv', 'ORDERS_WITH_STAFF_ASSIGNMENT_20260306.csv', 'BRONZE_posts_with_hours_20260306.csv', 'BRONZE_woocommerce_order_itemmeta_202503180953_20260306.csv', 'BRONZE_woocommerce_order_items_202503180953_20260306.csv', 'ORDERS_WITH_STAFF_ASSIGNMENT_20260306.csv']


In [5]:
import os
from config.paths import PATHS

os.listdir(PATHS["source"])

['BRONZE_posts_with_hours_20260306.csv',
 'BRONZE_woocommerce_order_itemmeta_202503180953_20260306.csv',
 'BRONZE_woocommerce_order_items_202503180953_20260306.csv',
 'ORDERS_WITH_STAFF_ASSIGNMENT_20260306.csv']

## 4. SILVER – order_items

In [6]:
from pipelines.transform_order_items import transform_order_items

order_items_bronze = read_parquet(f"{PATHS['bronze']}/order_items.parquet")
order_items_silver = transform_order_items(order_items_bronze)

order_items_silver.head()

,order_item_id,order_id,order_item_name,order_item_color,order_item_size
0,233,2767,Abrigos de paño,Gris,XS
1,1562,3398,Vestidos Formales de crepé satén,Azul marino,M
2,2026,3636,Blusas de satén,Camel,S
3,2027,3636,Cazadoras de piel,Gris,M
4,5072,5147,Blusas de viscosa,Camel,L


## 5. SILVER – order_itemmeta

In [7]:
from pipelines.transform_order_itemmeta import transform_order_itemmeta

order_itemmeta_bronze = read_parquet(f"{PATHS['bronze']}/order_itemmeta.parquet")
order_itemmeta_silver = transform_order_itemmeta(order_itemmeta_bronze)

order_itemmeta_silver.head()

meta_key,order_item_id,line_total,product_id,quantity
0,233,42363.0,3,1
1,1562,234666.0,2004,2
2,2026,35428.0,1020,1
3,2027,35216.0,543,1
4,5072,64935.0,1000,3


## 6. SILVER – orders_with_staff_assignment

In [8]:
from pipelines.transform_staff_assignment import transform_staff_assignment

orders_staff_bronze = read_parquet(f"{PATHS['bronze']}/orders_with_staff_assignment.parquet")
orders_staff_silver = transform_staff_assignment(orders_staff_bronze)

orders_staff_silver.head()

,order_id,store_name,staff_id,staff_name,num_items,total_value,fecha,assignment_timestamp,post_date
0,2767,Tienda Online,1005,Empleado Virtual 1005,1,42363.0,2026-03-06,2026-03-06,2026-03-06 00:00:00
1,3398,Tienda Online,1004,Empleado Virtual 1004,1,234666.0,2026-03-06,2026-03-06,2026-03-06 00:00:00
2,3636,Tienda Madrid Callao,30,Empleado 30,2,70644.0,2026-03-06,2026-03-06,2026-03-06 00:00:00
3,5147,Tienda Bilbao Ensanche,27,Empleado 27,3,263341.0,2026-03-06,2026-03-06,2026-03-06 00:00:00
4,10812,Tienda Barcelona Diagonal,25,Empleado 25,1,28505.0,2026-03-06,2026-03-06,2026-03-06 00:00:00


## 7. SILVER – post_with_hours

In [9]:
from pipelines.transform_post_with_hours import transform_post_with_hours

posthours_bronze = read_parquet(f"{PATHS['bronze']}/post_with_hours.parquet")
posthours_silver = transform_post_with_hours(posthours_bronze)

posthours_silver.head()

,post_author,post_date,post_title,post_status,fecha,order_id,order_timestamp
0,3758,2026-03-06 23:51:04,Pedido #2767,publish,2026-03-06 23:51:04,2767,2026-03-06 23:51:04
1,2967,2026-03-06 18:21:06,Pedido #3398,publish,2026-03-06 18:21:06,3398,2026-03-06 18:21:06
2,4,2026-03-06 21:19:30,Pedido #3636,publish,2026-03-06 21:19:30,3636,2026-03-06 21:19:30
3,4,2026-03-06 13:08:42,Pedido #5147,publish,2026-03-06 13:08:42,5147,2026-03-06 13:08:42
4,7,2026-03-06 13:31:45,Pedido #10812,publish,2026-03-06 13:31:45,10812,2026-03-06 13:31:45


## 8. GOLD – dim_staff + fact_staff_orders

In [10]:
from pipelines.gold_staff import create_dim_staff, create_fact_staff_orders

dim_staff = create_dim_staff(orders_staff_silver)
fact_staff_orders = create_fact_staff_orders(orders_staff_silver)

dim_staff.head(), fact_staff_orders.head()

(   staff_id             staff_name                 store_name  \
 0      1005  Empleado Virtual 1005              Tienda Online   
 1      1004  Empleado Virtual 1004              Tienda Online   
 2        30            Empleado 30       Tienda Madrid Callao   
 3        27            Empleado 27     Tienda Bilbao Ensanche   
 4        25            Empleado 25  Tienda Barcelona Diagonal   
 
         staff_name_clean first_seen_date last_seen_date  
 0  Empleado Virtual 1005      2026-03-06     2026-03-06  
 1  Empleado Virtual 1004      2026-03-06     2026-03-06  
 2            Empleado 30      2026-03-06     2026-03-06  
 3            Empleado 27      2026-03-06     2026-03-06  
 4            Empleado 25      2026-03-06     2026-03-06  ,
    order_id  staff_id assignment_timestamp  num_items  total_value  \
 0      2767      1005           2026-03-06          1      42363.0   
 1      3398      1004           2026-03-06          1     234666.0   
 2      3636        30           2

## 9. GOLD – dim_time + fact_orders_time

In [11]:
from pipelines.gold_time import create_dim_time, create_fact_orders_time

# Pass orders_staff_silver as the second argument so that assignment timestamps
# are included in dim_time — this eliminates the 11 orphan rows in
# fact_staff_orders → dim_time that appear when only order timestamps are used.
dim_time = create_dim_time(posthours_silver, orders_staff_silver)
fact_orders_time = create_fact_orders_time(posthours_silver)

# Orphan check: fact_staff_orders → dim_time (should be 0 after the fix above)
orphan_rows = fact_staff_orders.loc[
    ~fact_staff_orders["assignment_time_id"].isin(dim_time["time_id"])
]

print("🔎 Total orphans:", len(orphan_rows))
orphan_rows[["order_id", "staff_id", "assignment_timestamp", "assignment_time_id"]]

dim_time.head(), fact_orders_time.head()

🔎 Total huérfanos: 11


(      time_id                  ts  order_date  order_year  order_month  \
 0  2026030623 2026-03-06 23:51:04  2026-03-06        2026            3   
 1  2026030618 2026-03-06 18:21:06  2026-03-06        2026            3   
 2  2026030621 2026-03-06 21:19:30  2026-03-06        2026            3   
 3  2026030613 2026-03-06 13:08:42  2026-03-06        2026            3   
 4  2026030613 2026-03-06 13:31:45  2026-03-06        2026            3   
 
   order_month_name  order_day  order_hour order_hour_bucket  order_weekday  \
 0            March          6          23             noche              4   
 1            March          6          18             tarde              4   
 2            March          6          21             noche              4   
 3            March          6          13             tarde              4   
 4            March          6          13             tarde              4   
 
   order_weekday_name  is_weekend  
 0             Friday       False  


In [12]:
fact_staff_orders["assignment_time_id"].dtype, dim_time["time_id"].dtype

(Int64Dtype(), Int64Dtype())

In [13]:
print("dtypes:", fact_staff_orders["assignment_time_id"].dtype, dim_time["time_id"].dtype)

# Do the values actually match after casting?
test_ids = fact_staff_orders["assignment_time_id"].dropna().astype("Int64").unique()[:10]
print("Sample assignment_time_id values:", list(test_ids))

missing_ids = fact_staff_orders.loc[
    ~fact_staff_orders["assignment_time_id"].astype("Int64").isin(dim_time["time_id"].astype("Int64"))
, "assignment_time_id"].dropna().astype("Int64").unique()

print("Total orphans (by id):", len(missing_ids))
print("Missing IDs (up to 20):", list(missing_ids[:20]))

# Cross-check: do those IDs exist if we force the join?
tmp = fact_staff_orders[["assignment_time_id"]].drop_duplicates()
tmp["assignment_time_id"] = tmp["assignment_time_id"].astype("Int64")

dim_time_chk = dim_time[["time_id"]].drop_duplicates()
dim_time_chk["time_id"] = dim_time_chk["time_id"].astype("Int64")

anti = tmp.merge(dim_time_chk, left_on="assignment_time_id", right_on="time_id", how="left", indicator=True)
anti_missing = anti[anti["_merge"] == "left_only"]
print("Anti-join missing rows:", anti_missing.shape[0])
anti_missing.head(10)

dtypes: Int64 Int64
Ejemplo de assignment_time_id: [np.int64(2026030600)]
Total huérfanos (por id): 1
Missing IDs (hasta 20): [np.int64(2026030600)]
Anti-join faltantes: 1


,assignment_time_id,time_id,_merge
0,2026030600,<NA>,left_only


## 10. GOLD – dim_order

In [14]:
from pipelines.gold_dim_order import create_dim_order

dim_order = create_dim_order(orders_staff_silver)

dim_order.head()

,order_id,order_timestamp,order_date,time_id,store_name,num_items,total_value
0,2767,2026-03-06,2026-03-06,2026030600,Tienda Online,1,42363.0
1,3398,2026-03-06,2026-03-06,2026030600,Tienda Online,1,234666.0
2,3636,2026-03-06,2026-03-06,2026030600,Tienda Madrid Callao,2,70644.0
3,5147,2026-03-06,2026-03-06,2026030600,Tienda Bilbao Ensanche,3,263341.0
4,10812,2026-03-06,2026-03-06,2026030600,Tienda Barcelona Diagonal,1,28505.0


## 11. GOLD – fact_order_items

In [15]:
from pipelines.gold_models import create_fact_order_items

fact_order_items = create_fact_order_items(order_items_silver, order_itemmeta_silver)

fact_order_items.head()

,order_item_id,order_id,product_id,order_item_name,order_item_color,order_item_size,quantity,line_total,unit_price,total_with_tax
0,233,2767,3,Abrigos de paño,Gris,XS,1,42363.0,42363.0,42363.0
1,1562,3398,2004,Vestidos Formales de crepé satén,Azul marino,M,2,234666.0,117333.0,234666.0
2,2026,3636,1020,Blusas de satén,Camel,S,1,35428.0,35428.0,35428.0
3,2027,3636,543,Cazadoras de piel,Gris,M,1,35216.0,35216.0,35216.0
4,5072,5147,1000,Blusas de viscosa,Camel,L,3,64935.0,21645.0,64935.0


In [16]:
# order_itemmeta_silver.columns.tolist()

## 12. Full Pipeline Orchestration – main.py

In [17]:
# ============================================================
# FULL PIPELINE ORCHESTRATION
# ============================================================

import pandas as pd

# 1) INGESTION (SOURCE → BRONZE)
from pipelines.ingestion import ingest_all_source_csv
from utils.io import read_parquet
from config.paths import PATHS

print("\n==============================")
print("🚀 1) INGESTION (SOURCE → BRONZE)")
print("==============================")

ingested = ingest_all_source_csv(verbose=True)
print("\nFiles ingested into BRONZE:", ingested)



# ============================================================
# 2) SILVER – Transformations
# ============================================================

print("\n==============================")
print("🔧 2) SILVER – Transformations")
print("==============================")

# --- order_items ---
from pipelines.transform_order_items import transform_order_items
order_items_bronze = read_parquet(f"{PATHS['bronze']}/order_items.parquet")
order_items_silver = transform_order_items(order_items_bronze)

# --- order_itemmeta ---
from pipelines.transform_order_itemmeta import transform_order_itemmeta
order_itemmeta_bronze = read_parquet(f"{PATHS['bronze']}/order_itemmeta.parquet")
order_itemmeta_silver = transform_order_itemmeta(order_itemmeta_bronze)

# --- orders_with_staff_assignment ---
from pipelines.transform_staff_assignment import transform_staff_assignment
orders_staff_bronze = read_parquet(f"{PATHS['bronze']}/orders_with_staff_assignment.parquet")
orders_staff_silver = transform_staff_assignment(orders_staff_bronze)

# --- post_with_hours ---
from pipelines.transform_post_with_hours import transform_post_with_hours
posthours_bronze = read_parquet(f"{PATHS['bronze']}/post_with_hours.parquet")
posthours_silver = transform_post_with_hours(posthours_bronze)



# ============================================================
# 3) GOLD – Dimensions and Fact Tables
# ============================================================

print("\n==============================")
print("🏆 3) GOLD – Dimensions")
print("==============================")

# --- dim_staff ---
from pipelines.gold_staff import create_dim_staff, create_fact_staff_orders
dim_staff = create_dim_staff(orders_staff_silver)

# --- dim_time ---
from pipelines.gold_time import create_dim_time, create_fact_orders_time
# Pass orders_staff_silver to include assignment timestamps in dim_time
dim_time = create_dim_time(posthours_silver, orders_staff_silver)

# --- dim_order ---
from pipelines.gold_dim_order import create_dim_order
dim_order = create_dim_order(orders_staff_silver)


print("\n==============================")
print("📦 3B) GOLD – Fact tables")
print("==============================")

# --- fact_order_items ---
from pipelines.gold_models import create_fact_order_items
fact_order_items = create_fact_order_items(order_items_silver, order_itemmeta_silver)

# --- fact_staff_orders ---
fact_staff_orders = create_fact_staff_orders(orders_staff_silver)

# --- fact_orders_time ---
fact_orders_time = create_fact_orders_time(posthours_silver)



# ============================================================
# 4) REFERENTIAL INTEGRITY AUDIT
# ============================================================

print("\n==============================")
print("🔎 4) REFERENTIAL INTEGRITY AUDIT")
print("==============================")

from utils.io import read_parquet
import pandas as pd

def audit_orphans(fact_df, fact_key, dim_df, dim_key, label):
    missing = fact_df[~fact_df[fact_key].isin(dim_df[dim_key])]
    if len(missing) > 0:
        print(f"❌ {label}: {len(missing)} orphans found")
    else:
        print(f"✔ {label}: no orphans ✓")
    return missing

# Load all Gold tables
dim_order       = read_parquet(f"{PATHS['gold']}/dim_order.parquet")
dim_time        = read_parquet(f"{PATHS['gold']}/dim_time.parquet")
dim_staff       = read_parquet(f"{PATHS['gold']}/dim_staff.parquet")

fact_order_items  = read_parquet(f"{PATHS['gold']}/fact_order_items.parquet")
fact_orders_time  = read_parquet(f"{PATHS['gold']}/fact_orders_time.parquet")
fact_staff_orders = read_parquet(f"{PATHS['gold']}/fact_staff_orders.parquet")

# Run all orphan checks
o1 = audit_orphans(fact_order_items, "order_id", dim_order, "order_id",
                   "fact_order_items → dim_order")
o2 = audit_orphans(fact_orders_time, "order_id", dim_order, "order_id",
                   "fact_orders_time → dim_order")
o3 = audit_orphans(fact_staff_orders, "order_id", dim_order, "order_id",
                   "fact_staff_orders → dim_order")

t1 = audit_orphans(fact_orders_time, "time_id", dim_time, "time_id",
                   "fact_orders_time → dim_time")
t2 = audit_orphans(fact_staff_orders, "assignment_time_id", dim_time, "time_id",
                   "fact_staff_orders → dim_time")

s1 = audit_orphans(fact_staff_orders, "staff_id", dim_staff, "staff_id",
                   "fact_staff_orders → dim_staff")

print("\n==============================")
print("📌 AUDIT SUMMARY")
print("==============================")

pd.DataFrame([
    ("fact_order_items → dim_order", len(o1)),
    ("fact_orders_time → dim_order", len(o2)),
    ("fact_staff_orders → dim_order", len(o3)),
    ("fact_orders_time → dim_time", len(t1)),
    ("fact_staff_orders → dim_time", len(t2)),
    ("fact_staff_orders → dim_staff", len(s1)),
], columns=["Relationship", "Orphans"])


🚀 1) INGESTA (SOURCE → BRONZE)
[BRONZE] Source activo: source\2026\03\06
[BRONZE] Encontrados 4 fichero(s) CSV
[BRONZE] Ingestando: BRONZE_posts_with_hours_20260306.csv → bronze\2026\03\06\post_with_hours.parquet
[BRONZE] Ingestando: BRONZE_woocommerce_order_itemmeta_202503180953_20260306.csv → bronze\2026\03\06\order_itemmeta.parquet
[BRONZE] Ingestando: BRONZE_woocommerce_order_items_202503180953_20260306.csv → bronze\2026\03\06\order_items.parquet
[BRONZE] Ingestando: ORDERS_WITH_STAFF_ASSIGNMENT_20260306.csv → bronze\2026\03\06\orders_with_staff_assignment.parquet
[BRONZE] Ingesta completa. Total: 4 → ['post_with_hours', 'order_itemmeta', 'order_items', 'orders_with_staff_assignment']

Archivos ingeridos en BRONZE: ['post_with_hours', 'order_itemmeta', 'order_items', 'orders_with_staff_assignment']

🔧 2) SILVER – Transformaciones

🏆 3) GOLD – Dimensiones

📦 3B) GOLD – Hechos

🔎 4) AUDITORÍA DE INTEGRIDAD
✔ fact_order_items → dim_order: sin huérfanos ✓
✔ fact_orders_time → dim_orde

,Relación,Huérfanos
0,fact_order_items → dim_order,0
1,fact_orders_time → dim_order,0
2,fact_staff_orders → dim_order,0
3,fact_orders_time → dim_time,0
4,fact_staff_orders → dim_time,11
5,fact_staff_orders → dim_staff,0


## 13. Referential Integrity Audit (Fact → Dim)

In [18]:
from utils.io import read_parquet

def audit_orphans(fact_df, fact_key, dim_df, dim_key, label):
    missing = fact_df[~fact_df[fact_key].isin(dim_df[dim_key])]
    print(f"✔ {label}: no orphans") if missing.empty else print(f"❌ {label}: {len(missing)} orphans found")
    return missing

print("=== GOLD INTEGRITY AUDIT ===")

dim_order = read_parquet(f"{PATHS['gold']}/dim_order.parquet")
dim_time = read_parquet(f"{PATHS['gold']}/dim_time.parquet")
dim_staff = read_parquet(f"{PATHS['gold']}/dim_staff.parquet")

fact_order_items = read_parquet(f"{PATHS['gold']}/fact_order_items.parquet")
fact_orders_time = read_parquet(f"{PATHS['gold']}/fact_orders_time.parquet")
fact_staff_orders = read_parquet(f"{PATHS['gold']}/fact_staff_orders.parquet")

_foi = audit_orphans(fact_order_items, "order_id", dim_order, "order_id", "fact_order_items → dim_order")
_fot = audit_orphans(fact_orders_time, "order_id", dim_order, "order_id", "fact_orders_time → dim_order")
_fso = audit_orphans(fact_staff_orders, "order_id", dim_order, "order_id", "fact_staff_orders → dim_order")

_t_fot = audit_orphans(fact_orders_time, "time_id", dim_time, "time_id", "fact_orders_time → dim_time")
_t_fso = audit_orphans(fact_staff_orders, "assignment_time_id", dim_time, "time_id", "fact_staff_orders → dim_time")

_s = audit_orphans(fact_staff_orders, "staff_id", dim_staff, "staff_id", "fact_staff_orders → dim_staff")

=== AUDITORÍA DE INTEGRIDAD GOLD ===
✔ fact_order_items → dim_order sin huérfanos
✔ fact_orders_time → dim_order sin huérfanos
✔ fact_staff_orders → dim_order sin huérfanos
✔ fact_orders_time → dim_time sin huérfanos
❌ fact_staff_orders → dim_time: 11 huérfanos
✔ fact_staff_orders → dim_staff sin huérfanos


## 14. Gold Integrity Audit

In [19]:
# ============================================================
# REFERENTIAL INTEGRITY AUDIT ACROSS GOLD TABLES
# ============================================================

import pandas as pd
from utils.io import read_parquet
from config.paths import PATHS

def audit_orphans(fact_df, fact_key, dim_df, dim_key, label):
    """
    Returns orphan rows where the fact table FK has no matching row in the dimension.
    """
    missing = fact_df[~fact_df[fact_key].isin(dim_df[dim_key])]
    if len(missing) > 0:
        print(f"❌ {label}: {len(missing)} orphan records")
    else:
        print(f"✔ {label}: no orphans ✓")
    return missing


# Load Gold tables
dim_order = read_parquet(f"{PATHS['gold']}/dim_order.parquet")
dim_time  = read_parquet(f"{PATHS['gold']}/dim_time.parquet")
dim_staff = read_parquet(f"{PATHS['gold']}/dim_staff.parquet")

fact_order_items  = read_parquet(f"{PATHS['gold']}/fact_order_items.parquet")
fact_orders_time  = read_parquet(f"{PATHS['gold']}/fact_orders_time.parquet")
fact_staff_orders = read_parquet(f"{PATHS['gold']}/fact_staff_orders.parquet")


print("\n======================")
print("🔍 GOLD AUDIT")
print("======================\n")

# ============================================================
# 1. Orphan check: ORDER_ID → dim_order
# ============================================================

orphans_foi = audit_orphans(
    fact_order_items, "order_id",
    dim_order, "order_id",
    "fact_order_items → dim_order"
)

orphans_fot = audit_orphans(
    fact_orders_time, "order_id",
    dim_order, "order_id",
    "fact_orders_time → dim_order"
)

orphans_fso = audit_orphans(
    fact_staff_orders, "order_id",
    dim_order, "order_id",
    "fact_staff_orders → dim_order"
)


# ============================================================
# 2. Orphan check: TIME_ID → dim_time
# ============================================================

orphans_time_fot = audit_orphans(
    fact_orders_time, "time_id",
    dim_time, "time_id",
    "fact_orders_time → dim_time"
)

orphans_time_fso = audit_orphans(
    fact_staff_orders, "assignment_time_id",
    dim_time, "time_id",
    "fact_staff_orders → dim_time"
)


# ============================================================
# 3. Orphan check: STAFF_ID → dim_staff
# ============================================================

orphans_staff = audit_orphans(
    fact_staff_orders, "staff_id",
    dim_staff, "staff_id",
    "fact_staff_orders → dim_staff"
)


# ============================================================
# 4. Summary
# ============================================================

print("\n======================")
print("📌 ORPHAN SUMMARY")
print("======================\n")

summary = [
    ("fact_order_items → dim_order", len(orphans_foi)),
    ("fact_orders_time → dim_order", len(orphans_fot)),
    ("fact_staff_orders → dim_order", len(orphans_fso)),
    ("fact_orders_time → dim_time", len(orphans_time_fot)),
    ("fact_staff_orders → dim_time", len(orphans_time_fso)),
    ("fact_staff_orders → dim_staff", len(orphans_staff)),
]

pd.DataFrame(summary, columns=["Relationship", "Orphans"])


🔍 AUDITORÍA GOLD

✔ fact_order_items → dim_order: sin huérfanos ✓
✔ fact_orders_time → dim_order: sin huérfanos ✓
✔ fact_staff_orders → dim_order: sin huérfanos ✓
✔ fact_orders_time → dim_time: sin huérfanos ✓
❌ fact_staff_orders → dim_time: 11 registros huérfanos
✔ fact_staff_orders → dim_staff: sin huérfanos ✓

📌 RESUMEN DE ORFANDAD



,Relación,Huérfanos
0,fact_order_items → dim_order,0
1,fact_orders_time → dim_order,0
2,fact_staff_orders → dim_order,0
3,fact_orders_time → dim_time,0
4,fact_staff_orders → dim_time,11
5,fact_staff_orders → dim_staff,0


## Automated Pipeline Checklist

In [20]:
# ============================================================
# FULL PIPELINE CHECKLIST
# ============================================================

import os
import pandas as pd
from utils.io import read_parquet
from config.paths import PATHS


def check_file_exists(path):
    return os.path.exists(path)


def check_not_empty(df, label):
    if df is None:
        print(f"❌ {label}: DataFrame is None")
        return False
    if len(df) == 0:
        print(f"❌ {label}: DataFrame is empty")
        return False
    print(f"✔ {label}: OK ({len(df)} rows)")
    return True


def audit_orphans(fact_df, fact_key, dim_df, dim_key, label):
    missing = fact_df[~fact_df[fact_key].isin(dim_df[dim_key])]
    if len(missing) > 0:
        print(f"❌ {label}: {len(missing)} orphans found")
        return False
    else:
        print(f"✔ {label}: no orphans")
        return True


print("\n==============================")
print("🧪 PIPELINE CHECKLIST")
print("==============================\n")


# ============================================================
# 1) BRONZE – check expected files exist
# ============================================================

bronze_ok = True
bronze_expected = [
    "order_items.parquet",
    "order_itemmeta.parquet",
    "orders_with_staff_assignment.parquet",
    "post_with_hours.parquet"
]

print("📂 BRONZE CHECK:")
for fname in bronze_expected:
    path = f"{PATHS['bronze']}/{fname}"
    exists = check_file_exists(path)
    if exists:
        print(f"✔ {fname}: OK")
    else:
        print(f"❌ {fname}: NOT FOUND")
        bronze_ok = False

print()


# ============================================================
# 2) SILVER – verify tables are non-empty
# ============================================================

print("🔧 SILVER CHECK:\n")

silver_ok = True

try:
    order_items_silver = read_parquet(f"{PATHS['silver']}/order_items_silver.parquet")
    silver_ok &= check_not_empty(order_items_silver, "order_items_silver")
except:
    print("❌ ERROR loading order_items_silver")
    silver_ok = False

try:
    order_itemmeta_silver = read_parquet(f"{PATHS['silver']}/order_itemmeta_silver.parquet")
    silver_ok &= check_not_empty(order_itemmeta_silver, "order_itemmeta_silver")
except:
    print("❌ ERROR loading order_itemmeta_silver")
    silver_ok = False

try:
    orders_staff_silver = read_parquet(f"{PATHS['silver']}/orders_with_staff_assignment_silver.parquet")
    silver_ok &= check_not_empty(orders_staff_silver, "orders_with_staff_assignment_silver")
except:
    print("❌ ERROR loading orders_with_staff_assignment_silver")
    silver_ok = False

try:
    posthours_silver = read_parquet(f"{PATHS['silver']}/post_with_hours_silver.parquet")
    silver_ok &= check_not_empty(posthours_silver, "post_with_hours_silver")
except:
    print("❌ ERROR loading post_with_hours_silver")
    silver_ok = False

print()


# ============================================================
# 3) GOLD – verify dimensions and fact tables are non-empty
# ============================================================

print("🏆 GOLD CHECK:\n")

gold_ok = True
gold_tables = {
    "dim_order": "dim_order.parquet",
    "dim_time": "dim_time.parquet",
    "dim_staff": "dim_staff.parquet",
    "fact_order_items": "fact_order_items.parquet",
    "fact_orders_time": "fact_orders_time.parquet",
    "fact_staff_orders": "fact_staff_orders.parquet",
}

gold_loaded = {}

for name, fname in gold_tables.items():
    path = f"{PATHS['gold']}/{fname}"
    if not check_file_exists(path):
        print(f"❌ GOLD missing: {fname}")
        gold_ok = False
        continue
    try:
        df = read_parquet(path)
        ok = check_not_empty(df, name)
        gold_ok &= ok
        gold_loaded[name] = df
    except Exception as e:
        print(f"❌ ERROR reading {fname}: {e}")
        gold_ok = False

print()


# ============================================================
# 4) REFERENTIAL INTEGRITY AUDIT
# ============================================================

print("🔎 GOLD → DIM INTEGRITY CHECK:\n")

audit_ok = True

if "dim_order" in gold_loaded:
    dim_order = gold_loaded["dim_order"]
if "dim_time" in gold_loaded:
    dim_time = gold_loaded["dim_time"]
if "dim_staff" in gold_loaded:
    dim_staff = gold_loaded["dim_staff"]

if "fact_order_items" in gold_loaded:
    audit_ok &= audit_orphans(gold_loaded["fact_order_items"], "order_id", dim_order, "order_id",
                              "fact_order_items → dim_order")

if "fact_orders_time" in gold_loaded:
    audit_ok &= audit_orphans(gold_loaded["fact_orders_time"], "order_id", dim_order, "order_id",
                              "fact_orders_time → dim_order")
    audit_ok &= audit_orphans(gold_loaded["fact_orders_time"], "time_id", dim_time, "time_id",
                              "fact_orders_time → dim_time")

if "fact_staff_orders" in gold_loaded:
    audit_ok &= audit_orphans(gold_loaded["fact_staff_orders"], "order_id", dim_order, "order_id",
                              "fact_staff_orders → dim_order")
    audit_ok &= audit_orphans(gold_loaded["fact_staff_orders"], "assignment_time_id", dim_time, "time_id",
                              "fact_staff_orders → dim_time")
    audit_ok &= audit_orphans(gold_loaded["fact_staff_orders"], "staff_id", dim_staff, "staff_id",
                              "fact_staff_orders → dim_staff")


# ============================================================
# 5) FINAL RESULT
# ============================================================

print("\n==============================")
print("🏁 FINAL CHECKLIST RESULT")
print("==============================")

passed = bronze_ok and silver_ok and gold_ok and audit_ok

if passed:
    print("🎉 ALL CHECKS PASSED: The pipeline completed all phases without errors.")
else:
    print("⚠️ The pipeline did NOT pass all validations. Review the errors above.")

passed


🧪 CHECKLIST PIPELINE

📂 BRONZE CHECK:
✔ order_items.parquet: OK
✔ order_itemmeta.parquet: OK
✔ orders_with_staff_assignment.parquet: OK
✔ post_with_hours.parquet: OK

🔧 SILVER CHECK:

✔ order_items_silver: OK (21 filas)
✔ order_itemmeta_silver: OK (21 filas)
✔ orders_with_staff_assignment_silver: OK (11 filas)
✔ post_with_hours_silver: OK (11 filas)

🏆 GOLD CHECK:

✔ dim_order: OK (11 filas)
✔ dim_time: OK (11 filas)
✔ dim_staff: OK (10 filas)
✔ fact_order_items: OK (21 filas)
✔ fact_orders_time: OK (11 filas)
✔ fact_staff_orders: OK (11 filas)

🔎 AUDITORÍA GOLD → DIM CHECK:

✔ fact_order_items → dim_order: sin huérfanos
✔ fact_orders_time → dim_order: sin huérfanos
✔ fact_orders_time → dim_time: sin huérfanos
✔ fact_staff_orders → dim_order: sin huérfanos
❌ fact_staff_orders → dim_time: 11 huérfanos
✔ fact_staff_orders → dim_staff: sin huérfanos

🏁 RESULTADO FINAL DEL CHECKLIST
⚠️ El pipeline NO pasó todas las validaciones. Revisar errores arriba.


False

In [21]:
# Spot-check: list any remaining orphans after the dim_time fix.
# This should return an empty DataFrame once create_dim_time receives
# orders_staff_silver as its second argument.
fact = fact_staff_orders
dim = dim_time

orphan_ids = fact.loc[~fact["assignment_time_id"].isin(dim["time_id"]), :]
orphan_ids.head(20)

,order_id,staff_id,assignment_timestamp,num_items,total_value,assignment_time_id,orders_handled
0,2767,1005,2026-03-06,1,42363.0,2026030600,1
1,3398,1004,2026-03-06,1,234666.0,2026030600,1
2,3636,30,2026-03-06,2,70644.0,2026030600,1
3,5147,27,2026-03-06,3,263341.0,2026030600,1
4,10812,25,2026-03-06,1,28505.0,2026030600,1
5,11421,1006,2026-03-06,1,13613.0,2026030600,1
6,13293,32,2026-03-06,3,236702.0,2026030600,1
7,13451,15,2026-03-06,1,11631.0,2026030600,1
8,14508,18,2026-03-06,3,238977.0,2026030600,1
9,14597,30,2026-03-06,3,23233.0,2026030600,1
